In [5]:
import subprocess
import sys

packages = ['xgboost', 'scikit-learn']
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print("✓ XGBoost and scikit-learn installed (or already present)")

✓ XGBoost and scikit-learn installed (or already present)


In [6]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb

# --- STEP 1: LOAD DATA ---
FILE_NAME = 'energy_data_3_years.csv'
try:
    data = pd.read_csv(FILE_NAME)
except FileNotFoundError:
    print(f"Error: The file '{FILE_NAME}' was not found.")
    exit()

# Prepare data
data['Date'] = pd.to_datetime(data['Date'])
data = data.sort_values('Date').reset_index(drop=True)

print("Data loaded successfully!")
print(f"Date range: {data['Date'].min().date()} to {data['Date'].max().date()}")
print(f"Total records: {len(data)}")

# --- STEP 2: FEATURE ENGINEERING ---
print(f"\n{'='*70}")
print("FEATURE ENGINEERING FOR XGBOOST")
print(f"{'='*70}")

def engineer_features(df, target_col):
    """Create features for XGBoost forecasting"""
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').reset_index(drop=True)
    
    # Extract date features
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month
    df['Day'] = df['Date'].dt.day
    df['Quarter'] = df['Date'].dt.quarter
    df['DayOfYear'] = df['Date'].dt.dayofyear
    df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)
    df['DayOfWeek'] = df['Date'].dt.dayofweek  # 0=Monday, 6=Sunday
    df['IsWeekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)
    
    # Lag features
    for lag in [1, 7, 14, 30]:
        df[f'Lag{lag}'] = df[target_col].shift(lag)
    
    # Rolling statistics
    for window in [7, 14, 30]:
        df[f'RollingMean{window}'] = df[target_col].rolling(window=window, min_periods=1).mean()
        df[f'RollingStd{window}'] = df[target_col].rolling(window=window, min_periods=1).std()
    
    # Fill NaN values
    df = df.fillna(method='bfill')
    df = df.fillna(df[target_col].mean())
    
    return df

# --- STEP 3: PREPARE SOLAR DATA ---
print("\nEngineering features for Solar...")
solar_col = 'Solar_Energy_Produced_kWh'
solar_data = engineer_features(data[['Date', solar_col]].copy(), solar_col)

print(f"Mean: {solar_data[solar_col].mean():.2f} kWh")
print(f"Std: {solar_data[solar_col].std():.2f} kWh")
print(f"Features created: {solar_data.shape[1] - 2}")

# Prepare X, y
feature_cols = [col for col in solar_data.columns if col not in ['Date', solar_col]]
X_solar = solar_data[feature_cols]
y_solar = solar_data[solar_col]

# --- STEP 4: TRAIN SOLAR XGBOOST MODEL ---
print("\n" + "="*70)
print("SOLAR ENERGY - XGBOOST FORECASTING")
print("="*70)

print("\nTraining XGBoost model (Solar)...")
solar_model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='reg:squarederror',
    verbosity=0
)

solar_model.fit(X_solar, y_solar, verbose=False)
print("✓ Solar model trained successfully!")

# Feature importance
importance = solar_model.get_booster().get_score(importance_type='weight')
top_features = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:10]
print(f"\nTop 10 Important Features:")
for feature, score in top_features:
    print(f"  {feature}: {score}")

# --- STEP 5: FORECAST 2026 SOLAR ---
print("\nGenerating 2026 forecast...")

# Create forecast dates
forecast_dates = pd.date_range('2026-01-01', '2026-12-31', freq='D')
solar_forecast_list = []

# For each date in 2026
for date in forecast_dates:
    # Create feature row
    year = date.year
    month = date.month
    day = date.day
    quarter = (month - 1) // 3 + 1
    day_of_year = date.dayofyear
    week_of_year = date.isocalendar()[1]
    day_of_week = date.weekday()
    is_weekend = 1 if day_of_week in [5, 6] else 0
    
    # Get historical data up to this date
    hist_data = solar_data[solar_data['Date'] < date].copy()
    
    if len(hist_data) > 30:
        # Calculate lag and rolling features
        lag1 = hist_data[solar_col].iloc[-1] if len(hist_data) >= 1 else hist_data[solar_col].mean()
        lag7 = hist_data[solar_col].iloc[-7] if len(hist_data) >= 7 else hist_data[solar_col].mean()
        lag14 = hist_data[solar_col].iloc[-14] if len(hist_data) >= 14 else hist_data[solar_col].mean()
        lag30 = hist_data[solar_col].iloc[-30] if len(hist_data) >= 30 else hist_data[solar_col].mean()
        
        roll_mean_7 = hist_data[solar_col].tail(7).mean()
        roll_std_7 = hist_data[solar_col].tail(7).std()
        roll_mean_14 = hist_data[solar_col].tail(14).mean()
        roll_std_14 = hist_data[solar_col].tail(14).std()
        roll_mean_30 = hist_data[solar_col].tail(30).mean()
        roll_std_30 = hist_data[solar_col].tail(30).std()
    else:
        lag1 = lag7 = lag14 = lag30 = solar_data[solar_col].mean()
        roll_mean_7 = roll_mean_14 = roll_mean_30 = solar_data[solar_col].mean()
        roll_std_7 = roll_std_14 = roll_std_30 = solar_data[solar_col].std()
    
    # Create feature vector
    X_pred = pd.DataFrame([{
        'Year': year,
        'Month': month,
        'Day': day,
        'Quarter': quarter,
        'DayOfYear': day_of_year,
        'WeekOfYear': week_of_year,
        'DayOfWeek': day_of_week,
        'IsWeekend': is_weekend,
        'Lag1': lag1,
        'Lag7': lag7,
        'Lag14': lag14,
        'Lag30': lag30,
        'RollingMean7': roll_mean_7,
        'RollingStd7': roll_std_7,
        'RollingMean14': roll_mean_14,
        'RollingStd14': roll_std_14,
        'RollingMean30': roll_mean_30,
        'RollingStd30': roll_std_30
    }])
    
    # Predict
    pred = solar_model.predict(X_pred)[0]
    solar_forecast_list.append(max(pred, 0))  # Ensure non-negative

print(f"✓ Forecast generated!")
print(f"\nSolar Forecast Statistics (2026):")
print(f"Mean: {np.mean(solar_forecast_list):.2f} kWh")
print(f"Std: {np.std(solar_forecast_list):.2f} kWh")
print(f"Min: {np.min(solar_forecast_list):.2f} kWh")
print(f"Max: {np.max(solar_forecast_list):.2f} kWh")

# Create output dataframe
forecast_df = pd.DataFrame({
    'Date': forecast_dates,
    'Solar_Energy_Forecast_kWh': solar_forecast_list
})

# --- STEP 6: WIND ENERGY FORECAST ---
wind_col = 'Wind_Energy_Produced_kWh'
if wind_col in data.columns:
    print(f"\n{'='*70}")
    print("WIND ENERGY - XGBOOST FORECASTING")
    print(f"{'='*70}")
    
    print("\nEngineering features for Wind...")
    wind_data = engineer_features(data[['Date', wind_col]].copy(), wind_col)
    
    print(f"Mean: {wind_data[wind_col].mean():.2f} kWh")
    print(f"Std: {wind_data[wind_col].std():.2f} kWh")
    
    # Prepare X, y
    X_wind = wind_data[feature_cols]
    y_wind = wind_data[wind_col]
    
    # Train wind model
    print("\nTraining XGBoost model (Wind)...")
    wind_model = xgb.XGBRegressor(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        objective='reg:squarederror',
        verbosity=0
    )
    
    wind_model.fit(X_wind, y_wind, verbose=False)
    print("✓ Wind model trained successfully!")
    
    # Forecast wind
    print("\nGenerating wind forecast...")
    wind_forecast_list = []
    
    for date in forecast_dates:
        year = date.year
        month = date.month
        day = date.day
        quarter = (month - 1) // 3 + 1
        day_of_year = date.dayofyear
        week_of_year = date.isocalendar()[1]
        day_of_week = date.weekday()
        is_weekend = 1 if day_of_week in [5, 6] else 0
        
        hist_data = wind_data[wind_data['Date'] < date].copy()
        
        if len(hist_data) > 30:
            lag1 = hist_data[wind_col].iloc[-1] if len(hist_data) >= 1 else hist_data[wind_col].mean()
            lag7 = hist_data[wind_col].iloc[-7] if len(hist_data) >= 7 else hist_data[wind_col].mean()
            lag14 = hist_data[wind_col].iloc[-14] if len(hist_data) >= 14 else hist_data[wind_col].mean()
            lag30 = hist_data[wind_col].iloc[-30] if len(hist_data) >= 30 else hist_data[wind_col].mean()
            
            roll_mean_7 = hist_data[wind_col].tail(7).mean()
            roll_std_7 = hist_data[wind_col].tail(7).std()
            roll_mean_14 = hist_data[wind_col].tail(14).mean()
            roll_std_14 = hist_data[wind_col].tail(14).std()
            roll_mean_30 = hist_data[wind_col].tail(30).mean()
            roll_std_30 = hist_data[wind_col].tail(30).std()
        else:
            lag1 = lag7 = lag14 = lag30 = wind_data[wind_col].mean()
            roll_mean_7 = roll_mean_14 = roll_mean_30 = wind_data[wind_col].mean()
            roll_std_7 = roll_std_14 = roll_std_30 = wind_data[wind_col].std()
        
        X_pred = pd.DataFrame([{
            'Year': year,
            'Month': month,
            'Day': day,
            'Quarter': quarter,
            'DayOfYear': day_of_year,
            'WeekOfYear': week_of_year,
            'DayOfWeek': day_of_week,
            'IsWeekend': is_weekend,
            'Lag1': lag1,
            'Lag7': lag7,
            'Lag14': lag14,
            'Lag30': lag30,
            'RollingMean7': roll_mean_7,
            'RollingStd7': roll_std_7,
            'RollingMean14': roll_mean_14,
            'RollingStd14': roll_std_14,
            'RollingMean30': roll_mean_30,
            'RollingStd30': roll_std_30
        }])
        
        pred = wind_model.predict(X_pred)[0]
        wind_forecast_list.append(max(pred, 0))
    
    forecast_df['Wind_Energy_Forecast_kWh'] = wind_forecast_list
    
    print(f"✓ Wind forecast completed!")
    print(f"Mean: {np.mean(wind_forecast_list):.2f} kWh")
    print(f"Max: {np.max(wind_forecast_list):.2f} kWh")

# --- STEP 7: SAVE AND DISPLAY RESULTS ---
print(f"\n{'='*70}")
print("FORECAST RESULTS")
print(f"{'='*70}")

# Save forecast
output_file = 'xgboost_forecast_2026.csv'
forecast_df.to_csv(output_file, index=False)
print(f"✓ Forecast saved to '{output_file}'")

# Display sample
print(f"\nForecast Sample (First 10 days):")
print(forecast_df.head(10).to_string(index=False))

# Monthly summary
forecast_df['Month'] = forecast_df['Date'].dt.to_period('M')
monthly = forecast_df.groupby('Month')['Solar_Energy_Forecast_kWh'].agg(['mean', 'min', 'max', 'sum'])

print(f"\n{'='*70}")
print("MONTHLY SUMMARY (2026)")
print(f"{'='*70}")
print(monthly.to_string())

# Total energy
total_solar = forecast_df['Solar_Energy_Forecast_kWh'].sum()
print(f"\nTotal Solar Energy (2026): {total_solar:,.2f} kWh")

if 'Wind_Energy_Forecast_kWh' in forecast_df.columns:
    total_wind = forecast_df['Wind_Energy_Forecast_kWh'].sum()
    total_combined = total_solar + total_wind
    print(f"Total Wind Energy (2026): {total_wind:,.2f} kWh")
    print(f"Combined Total (2026): {total_combined:,.2f} kWh")
    print(f"\nSolar: {(total_solar/total_combined)*100:.1f}%")
    print(f"Wind: {(total_wind/total_combined)*100:.1f}%")

print(f"\n{'='*70}")
print("✓ XGBOOST FORECASTING COMPLETE")
print(f"{'='*70}")

Data loaded successfully!
Date range: 2022-12-10 to 2025-12-08
Total records: 1095

FEATURE ENGINEERING FOR XGBOOST

Engineering features for Solar...
Mean: 9.53 kWh
Std: 2.01 kWh
Features created: 18

SOLAR ENERGY - XGBOOST FORECASTING

Training XGBoost model (Solar)...
✓ Solar model trained successfully!

Top 10 Important Features:
  RollingStd7: 881.0
  Lag1: 834.0
  Day: 770.0
  Lag7: 720.0
  RollingMean7: 714.0
  RollingStd14: 654.0
  Lag14: 646.0
  Lag30: 615.0
  RollingStd30: 531.0
  DayOfYear: 523.0

Generating 2026 forecast...
✓ Solar model trained successfully!

Top 10 Important Features:
  RollingStd7: 881.0
  Lag1: 834.0
  Day: 770.0
  Lag7: 720.0
  RollingMean7: 714.0
  RollingStd14: 654.0
  Lag14: 646.0
  Lag30: 615.0
  RollingStd30: 531.0
  DayOfYear: 523.0

Generating 2026 forecast...
✓ Forecast generated!

Solar Forecast Statistics (2026):
Mean: 6.87 kWh
Std: 0.09 kWh
Min: 6.53 kWh
Max: 7.28 kWh

WIND ENERGY - XGBOOST FORECASTING

Engineering features for Wind...
Mean: